In [1]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

# `ArrayList` vs. `LinkedList`

| operation | `ArrayList` | `LinkedList` |
|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(N)` |
| `add(e)` (back) | `O(1)` amortized | `O(1)` |
| `addFirst(e)`, `add(0, e)` | `O(N)` | `O(1)` |
| `removeLast()`, `remove(size - 1)` | `O(1)` | `O(1)` |
| `removeFirst()`, `remove(0)` | `O(N)` | `O(1)` |
| `add(i, e)`, `remove(i)` (middle) | `O(N)` | `O(N)` |
| `remove` / `add` through an iterator | `O(N)` | `O(1)` |
| `indexOf`, `contains` | `O(N)` | `O(N)` |
| memory per element | one array cell | one node: data, two references, object header |

Inserting in the middle is `O(N)` for both. Which one is faster?

In [2]:
System.out.printf("%8s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 10_000; n <= 40_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        List<Integer> l = new ArrayList<>();
        for (int i = 0; i < N; i++) l.add(l.size() / 2, i);
    });
    long lnk = millis(() -> {
        List<Integer> l = new LinkedList<>();
        for (int i = 0; i < N; i++) l.add(l.size() / 2, i);
    });
    System.out.printf("%,8d %10d %10d%n", N, arr, lnk);
}

       N   array ms  linked ms
  10,000          6        107
  20,000         14        389
  40,000         53       1514


- Same `O(N)`: where does the difference come from? Shifting a block of consecutive cells vs. following `N/2` references to nodes scattered in memory.
- In practice: `ArrayList` is the default choice. `LinkedList` when the work happens at the ends, or through an iterator.

# `String` vs. `StringBuilder`

## `String` is immutable

- No method changes a `String`: `concat`, `toUpperCase`, `+` return a **new** one.
- `s += x`: a new string, all of `s` copied, then `x`. Cost: `O(length of s)`.

```java
String s = "";
for (int i = 0; i < N; i++)
    s += i % 10;        // copies the i characters built so far
```

Cost of the loop?

## `StringBuilder`

- An array of characters with spare capacity, plus a count, so basically an ArrayList for characters
- Full: a new array of about double the capacity, copy
- `append`: `O(1)` amortized. `toString()`: one copy at the end, `O(N)`
- Also: `insert(pos, x)`, `deleteCharAt(pos)`, `setCharAt(pos, c)`, `reverse()`, `length()`

In [3]:
StringBuilder sb = new StringBuilder("list");
sb.append(": ").append(3.5).append(", ").append(true);     // append returns sb itself
System.out.println(sb);

sb.insert(0, '[').append(']');
sb.setCharAt(1, 'L');
System.out.println(sb + " " + sb.length());

list: 3.5, true
[List: 3.5, true] 17


In [4]:
System.out.printf("%8s %10s %10s%n", "N", "+= ms", "append ms");
for (int n = 25_000; n <= 200_000; n *= 2) {
    final int N = n;
    long str = millis(() -> {
        String s = "";
        for (int i = 0; i < N; i++) s += i % 10;
    });
    long bld = millis(() -> {
        StringBuilder b = new StringBuilder();
        for (int i = 0; i < N; i++) b.append(i % 10);
        String s = b.toString();
    });
    System.out.printf("%,8d %10d %10d%n", N, str, bld);
}

       N      += ms  append ms
  25,000         86          1
  50,000        253          3
 100,000        900          2
 200,000       3490          2


- Which column grows ×4 when `N` doubles?
- `"x = " + x + ", y = " + y` in one expression: a problem?
- `sb.insert(0, c)` in a loop of `N` iterations: cost?
- `toString` in `MyLinkedListDouble`: why a `StringBuilder`?

# Stacks

## Stack ADT

- Elements are added and removed at the **same end**: the **top**.
- **LIFO**: last in, first out.
- Examples: undo in an editor, back button in a browser, the call stack.

| operation | |
|---|---|
| `push(e)` | add `e` on top |
| `pop()` | remove and return the top element |
| `peek()` | return the top element, without removing it |
| `isEmpty()`, `size()` | |

- No `get(i)`, no `add(i, e)`: only the top is accessible.
- A list can do everything a stack does. Why define a separate ADT?

In [5]:
interface MyStack<E> {
    void push(E e);

    // throw NoSuchElementException if the stack is empty
    E pop();
    E peek();

    boolean isEmpty();
    int size();
}

### Popping an empty stack

- Our choice: throw `NoSuchElementException`.
- Alternative: return `null`. What happens after `push(null)`?

## Array-based stack

- Elements in `data[0..size-1]`, as in `MyArrayListDouble`.
- Where is the top: index `0` or index `size - 1`? Cost of `push` and `pop` in each case?

Top at `size - 1`:
- next free cell: `data[size]`
- top element: `data[size - 1]`
- full array: double the capacity

### Generic array

- `new E[4]` does not compile (erasure).
- `(E[]) new Object[4]` instead, with `@SuppressWarnings("unchecked")`.

In [6]:
class ArrayStack<E> implements MyStack<E> {
    private E[] data;
    private int size;       // elements in data[0..size-1], top at size-1

    @SuppressWarnings("unchecked")
    ArrayStack() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    public void push(E e) {
        if (size == data.length)
            data = Arrays.copyOf(data, 2 * data.length);
        data[size] = e;
        size++;
    }

    public E pop() {
        if (size == 0) throw new NoSuchElementException("empty stack");
        size--;
        E e = data[size];
        data[size] = null;
        return e;
    }

    public E peek() {
        if (size == 0) throw new NoSuchElementException("empty stack");
        return data[size - 1];
    }

    // bottom to top: the top is the rightmost element
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[i]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [7]:
MyStack<String> s = new ArrayStack<>();
s.push("a");
s.push("b");
s.push("c");
System.out.println(s + "  peek=" + s.peek() + "  size=" + s.size());

for (String x : List.of("d", "e", "f")) s.push(x);     // grows from 4 to 8
System.out.println(s);

while (!s.isEmpty())
    System.out.print(s.pop() + " ");
System.out.println();

try {
    s.pop();
} catch (NoSuchElementException e) {
    System.out.println(e);
}

[a, b, c]  peek=c  size=3
[a, b, c, d, e, f]
f e d c b a 
java.util.NoSuchElementException: empty stack


- `data[size] = null` in `pop`: the stack works without it. Why is it there?
  - `MyArrayListDouble` did not need it: why?
- Cost of `push`? Of `pop`, `peek`?

## Linked stack

- A chain of nodes, as in `MyLinkedListDouble`.
- Where is the top: first node or last node?

| top at | `push` | `pop` |
|---|---|---|
| first node (`head`) | ? | ? |
| last node (`tail`) | ? | ? |

Top at the first node:
- `push`: `addFirst`
- `pop`: `removeFirst`
- No `tail` needed. We call `head` `top`.

```java
public void push(E e) {
    top = new Node<>(e, top);       // the new node points to the old top
    size++;
}
```

Does it work on an empty stack?

In [8]:
class LinkedStack<E> implements MyStack<E> {
    private static class Node<E> {
        E data;
        Node<E> next;       // the node below this one

        Node(E data, Node<E> next) {
            this.data = data;
            this.next = next;
        }
    }

    private Node<E> top;    // null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return top == null; }

    public void push(E e) {
        top = new Node<>(e, top);
        size++;
    }

    public E pop() {
        if (top == null) throw new NoSuchElementException("empty stack");
        E e = top.data;
        top = top.next;
        size--;
        return e;
    }

    public E peek() {
        if (top == null) throw new NoSuchElementException("empty stack");
        return top.data;
    }

    // top to bottom
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("top: ");
        for (Node<E> n = top; n != null; n = n.next)
            sb.append(n.data).append(" ");
        return sb.toString();
    }
}

In [9]:
MyStack<Integer> t = new LinkedStack<>();
for (int i = 1; i <= 5; i++) t.push(i * i);
System.out.println(t + "  size=" + t.size());
System.out.println(t.pop() + " " + t.pop() + " " + t.peek());
System.out.println(t + "  size=" + t.size());

top: 25 16 9 4 1   size=5
25 16 9
top: 9 4 1   size=3


- Popped node: who frees it?
- Same `MyStack` interface: the code using the stack does not change. Only the `new` does.

## Array vs. linked stack

| | array | linked |
|---|---|---|
| `push` | `O(1)` amortized, `O(N)` worst case | `O(1)` |
| `pop`, `peek` | `O(1)` | `O(1)` |
| memory | one cell per element, plus unused capacity | one node per element |

In [10]:
System.out.printf("%10s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 1_000_000; n <= 8_000_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        MyStack<Integer> s = new ArrayStack<>();
        for (int i = 0; i < N; i++) s.push(i % 10);    // small Integers are cached: no allocation
        while (!s.isEmpty()) s.pop();
    });
    long lnk = millis(() -> {
        MyStack<Integer> s = new LinkedStack<>();
        for (int i = 0; i < N; i++) s.push(i % 10);
        while (!s.isEmpty()) s.pop();
    });
    System.out.printf("%,10d %10d %10d%n", N, arr, lnk);
}

         N   array ms  linked ms
 1,000,000         16         16
 2,000,000         20         17
 4,000,000         38        152
 8,000,000         62        349


- Both columns roughly double with `N`: expected?
- Same `O(1)` per operation: where does the gap come from? Compare with `ArrayList` vs. `LinkedList` above.
- A program needs every `push` to take the same time (no occasional `O(N)`): which one?

# Balanced delimiters

## The problem

Do the `( )`, `[ ]`, `{ }` in a string match?

| string | balanced? |
|---|---|
| `{[()]}` | yes |
| `a[i] = f(b[j]);` | yes |
| `([)]` | no |
| `(()` | no |
| `())` | no |

- A closing delimiter must match the **most recent** opening one that is still unmatched.
- Most recent: the top of a stack.

## Algorithm

For each character `c`:
- opening: `push(c)`
- closing: the stack must not be empty, and `pop()` must be the matching opening one
- anything else: skip

At the end: balanced if and only if the stack is empty.

In [11]:
boolean balanced(String s) {
    MyStack<Character> open = new ArrayStack<>();
    for (char c : s.toCharArray()) {
        if (c == '(' || c == '[' || c == '{') {
            open.push(c);
        } else if (c == ')' || c == ']' || c == '}') {
            if (open.isEmpty()) return false;
            char o = open.pop();
            if (!(o == '(' && c == ')' || o == '[' && c == ']' || o == '{' && c == '}'))
                return false;
        }
    }
    return open.isEmpty();
}

for (String s : List.of("{[()]}", "a[i] = f(b[j]);", "([)]", "(()", "())", ""))
    System.out.printf("%-18s %b%n", "\"" + s + "\"", balanced(s));

"{[()]}"           true
"a[i] = f(b[j]);"  true
"([)]"             false
"(()"              false
"())"              false
""                 true


- `([)]`, `(()`, `())` fail for three different reasons. Which line catches each?
- Cost? Largest possible size of the stack?
- Only `(` and `)`: is a stack needed? What would replace it?
- Three counters, one per type of delimiter: enough?

## Exercises

### 1. Tracing
Empty `ArrayStack<Integer>`, capacity 4. Draw `data` and give `size` after each operation. What do `pop` and `peek` return?

```java
s.push(5);
s.push(8);
s.push(2);
s.pop();
s.push(7);
s.peek();
s.push(1);
s.push(9);      // what happens here?
s.pop();
s.pop();
```

### 2. The other end
- Array stack with the top at index `0`: cost of `push` and `pop`?
- Linked stack with the top at the last node, with `head` and `tail`: cost of `push` and `pop`?
- Same, but doubly linked?

In [12]:
// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

### 3. A stack on top of a list
`ListStack<E> implements MyStack<E>`, with a single field `ArrayList<E> list`. Each method in one line. Which end of the list is the top?

In [13]:
MyStack<Integer> ls = new ListStack<>();
check(ls.isEmpty(), "empty at start");
ls.push(1);
ls.push(2);
ls.push(3);
check(ls.size() == 3, "size");
check(ls.pop() == 3 && ls.peek() == 2 && ls.size() == 2, "pop, peek");

CompilationException: 

### 4. Bottom to top
Change `toString` in `LinkedStack` so it matches `ArrayStack`: `[1, 2, 3]` after pushing `1`, `2`, `3`.
- The nodes go from top to bottom. With `sb.insert(0, ...)`: cost?
- Write an `O(N)` version.

In [ ]:
LinkedStack<Integer> lk = new LinkedStack<>();
check(lk.toString().equals("[]"), "empty: " + lk);
lk.push(1);
lk.push(2);
lk.push(3);
check(lk.toString().equals("[1, 2, 3]"), "three elements: " + lk);

### 5. Copying a stack
`<E> MyStack<E> copy(MyStack<E> s)`: a new `ArrayStack` with the same elements in the same order. Use only the `MyStack` methods. At the end, `s` must be as it was.

In [ ]:
MyStack<String> orig = new LinkedStack<>();
orig.push("x");
orig.push("y");
orig.push("z");
MyStack<String> cp = copy(orig);
check(cp.toString().equals("[x, y, z]"), "same order: " + cp);
check(orig.size() == 3 && orig.peek().equals("z"), "original unchanged");
cp.pop();
check(orig.size() == 3, "independent of the original");

### 6. Minimum in `O(1)`
`MinStack implements MyStack<Integer>`, plus `int min()`: the smallest element in the stack. All methods `O(1)`.

Hint: what is the minimum after a `pop`? Keep one for every element.

In [ ]:
MinStack ms = new MinStack();
ms.push(5);
ms.push(3);
ms.push(7);
ms.push(3);
ms.push(1);
check(ms.min() == 1, "min 1");
ms.pop();
check(ms.min() == 3, "min 3");
ms.pop();
check(ms.min() == 3, "min still 3: duplicates");
ms.pop();
ms.pop();
check(ms.min() == 5, "min 5");

### 7. Iterating a stack
Make `LinkedStack<E>` also implement `Iterable<E>`, from top to bottom.
- Which order makes sense for `ArrayStack`? Is it the order of `toString`?

In [ ]:
LinkedStack<Integer> it7 = new LinkedStack<>();
for (int i = 1; i <= 4; i++) it7.push(i);
StringBuilder out = new StringBuilder();
for (int x : it7) out.append(x);
check(out.toString().equals("4321"), "top to bottom: " + out);
check(it7.size() == 4, "iterating does not pop");

### 8. Where is the error?
`int firstError(String s)`: `-1` if `s` is balanced, otherwise the position of the first character that makes it unbalanced. If opening delimiters are left unclosed, the position of the first one.

Hint: what should the stack hold?

In [ ]:
check(firstError("{[()]}") == -1, "balanced");
check(firstError("([)]") == 2, "wrong closing");
check(firstError("())") == 2, "closing with nothing open");
check(firstError("x(y[z]") == 1, "never closed");
check(firstError("(()") == 0, "never closed, the outer one");